In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import joblib

# Load cleaned data
df = pd.read_csv('/Users/sunandangarg/Desktop/loan_project/loansense-mono/apps/api/data/cleaned_full.csv')

print("Shape:", df.shape)
print("Columns:", df.columns.tolist()[:10])
print("Default rate:", df['target'].mean().round(3))

Shape: (367693, 69)
Columns: ['loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'emp_length', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low']
Default rate: 0.198


In [2]:
from sklearn.preprocessing import StandardScaler

# Prepare features
X = df.drop('target', axis=1).values
y = df['target'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Reshape for LSTM: (samples, timesteps, features)
# We treat each feature group as a timestep
X_3d = X_scaled.reshape(X_scaled.shape[0], 1, X_scaled.shape[1])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_3d, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# Dataset class
class LoanDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = LoanDataset(X_train, y_train)
test_dataset = LoanDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512)

# LSTM Model
class LoanLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super(LoanLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        out = lstm_out[:, -1, :]
        return self.classifier(out).squeeze()

# Initialize
input_size = X_train.shape[2]
lstm_model = LoanLSTM(input_size=input_size)
print(f"\nModel parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")

# Training
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

# Train for 10 epochs
for epoch in range(10):
    lstm_model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = lstm_model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Evaluate
    lstm_model.eval()
    all_preds = []
    with torch.no_grad():
        for X_batch, _ in test_loader:
            preds = lstm_model(X_batch)
            all_preds.extend(preds.numpy())

    auc = roc_auc_score(y_test, all_preds)
    scheduler.step()
    print(f"Epoch {epoch+1}/10 | Loss: {total_loss/len(train_loader):.4f} | AUC: {auc:.4f}")

print(f"\nFinal LSTM AUC: {auc:.4f}")

Train shape: (294154, 1, 68)
Test shape: (73539, 1, 68)

Model parameters: 69,697
Epoch 1/10 | Loss: 0.4649 | AUC: 0.7401
Epoch 2/10 | Loss: 0.4401 | AUC: 0.7418
Epoch 3/10 | Loss: 0.4392 | AUC: 0.7426
Epoch 4/10 | Loss: 0.4382 | AUC: 0.7431
Epoch 5/10 | Loss: 0.4371 | AUC: 0.7433
Epoch 6/10 | Loss: 0.4369 | AUC: 0.7433
Epoch 7/10 | Loss: 0.4362 | AUC: 0.7436
Epoch 8/10 | Loss: 0.4358 | AUC: 0.7437
Epoch 9/10 | Loss: 0.4358 | AUC: 0.7436
Epoch 10/10 | Loss: 0.4353 | AUC: 0.7437

Final LSTM AUC: 0.7437


In [3]:
import gc
import torch
import numpy as np
from sklearn.metrics import roc_auc_score
import joblib

# Clear memory first
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

# Get LSTM predictions on test set (already in memory)
lstm_model.eval()
lstm_preds = []
with torch.no_grad():
    for X_batch, _ in test_loader:
        preds = lstm_model(X_batch)
        lstm_preds.extend(preds.cpu().numpy())
lstm_preds = np.array(lstm_preds)

print(f"LSTM AUC: {roc_auc_score(y_test, lstm_preds):.4f}")

# Save LSTM and scaler
torch.save(lstm_model.state_dict(), '/Users/sunandangarg/Desktop/loan_project/loansense-mono/apps/api/models/lstm_v1.pt')
joblib.dump(scaler, '/Users/sunandangarg/Desktop/loan_project/loansense-mono/apps/api/models/scaler.joblib')

print("Saved!")

LSTM AUC: 0.7437
Saved!


In [4]:
import numpy as np
lstm_preds = np.array(preds)
np.save('/Users/sunandangarg/Desktop/loan_project/loansense-mono/apps/api/models/lstm_preds.npy', lstm_preds)
np.save('/Users/sunandangarg/Desktop/loan_project/loansense-mono/apps/api/models/y_test.npy', y_test)
print("LSTM preds saved!", lstm_preds.shape)

LSTM preds saved! (323,)


/var/folders/c9/qlh3l8h12zq9p0_xgs9429d00000gn/T/ipykernel_96823/35001265.py:2: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  lstm_preds = np.array(preds)


In [5]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Load only 50K rows
df = pd.read_csv('/Users/sunandangarg/Desktop/loan_project/loansense-mono/apps/api/data/cleaned_full.csv', nrows=50000)
X = df.drop('target', axis=1).values
y = df['target'].values

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# XGBoost
xgb_model = joblib.load('/Users/sunandangarg/Desktop/loan_project/loansense-mono/apps/api/models/xgboost_v1.joblib')
xgb_preds = xgb_model.predict_proba(X_test)[:, 1]
print("XGBoost AUC:", roc_auc_score(y_test, xgb_preds).round(4))

np.save('/Users/sunandangarg/Desktop/loan_project/loansense-mono/apps/api/models/xgb_preds.npy', xgb_preds)
np.save('/Users/sunandangarg/Desktop/loan_project/loansense-mono/apps/api/models/y_test.npy', y_test)
print("Saved! Shape:", xgb_preds.shape)

: 